In [103]:
import pandas as pd
import matplotlib.pyplot as plt

path = r"D:/Projet 4/TaxiOut AI/data/cleaned_data.parquet"

df = pd.read_parquet(path)



------------------------------------VERIFIER LES COLONNES--------------------------

In [104]:
print(df.columns.tolist())

['month', 'day_of_month', 'scheduled_departure_time', 'scheduled_arrival_time', 'unique_carrier', 'flight_number', 'scheduled_elapsed_time', 'departure_delay', 'origin', 'dest', 'distance', 'taxi_out', 'cancelled', 'diverted', 'date']


-------------------------------FEATURE TEMPORELLES----------------------------------

In [105]:

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week_num"] = df["date"].dt.dayofweek


------------------------AJOUTER FEATURE WEEKEND----------------------

In [106]:
df["is_weekend"] = df["day_of_week_num"].isin([5, 6]).astype(int)

------------------EXTRAIRE HEURE DE DEPART----------------------

In [107]:
df["departure_hour"] = df["scheduled_departure_time"] // 100

-----------------FEATURE HEURE DE POINTE---------------------

In [108]:
peak_hours = [7,8,9,16,17,18,19]

df["is_peak_hour"] = df["departure_hour"].isin(peak_hours).astype(int)

-------------AEROPORT TRAFIQUE------------------

In [109]:
airport_traffic = df.groupby("origin").size()

df["airport_traffic"] = df["origin"].map(airport_traffic)

---------------------------------------------CARRIERE TRAFIQUE-=------------------------------------

In [110]:
carrier_traffic = df.groupby("unique_carrier").size()

df["carrier_traffic"] = df["unique_carrier"].map(carrier_traffic)

---------------------------------------DISTANCE FEATURE---------------------------------------

In [111]:
df["short_flight"] = (df["distance"] < 1000).astype(int)

---------------NOUVELLE COLONNES-------------------------

In [112]:
print(df.head())

   month  day_of_month  scheduled_departure_time  scheduled_arrival_time  \
0      1             4                      1125                    1240   
1      1             4                       545                     880   
2      1             4                       875                    1030   
3      1             4                       415                     570   
4      1             4                      1215                    1360   

  unique_carrier  flight_number  scheduled_elapsed_time  departure_delay  \
0             WN            746                    55.0             78.0   
1             WN           2126                   215.0              1.0   
2             WN             45                    95.0              9.0   
3             WN             87                    95.0              1.0   
4             WN            230                    85.0            108.0   

  origin dest  ...       date  year  day  day_of_week_num is_weekend  \
0    ABQ  AMA 

-----------------------------DATE DE DEPART--------------------------------------------------------

In [113]:
df["scheduled_departure_datetime"] = pd.to_datetime(df["date"])

----------------------------HEURE  ET MINUTE DE DEPART-------------------------------------------------------------

In [114]:
df["departure_hour"] = (
    df["scheduled_departure_time"] // 100
)

df["departure_minute"] = (
    df["scheduled_departure_time"] % 100
)

--------------------------CONSTRUIRE DATE TIME COMPLET-------------------------------------------------------------------------

In [115]:
df["scheduled_departure_datetime"] = (
    df["scheduled_departure_datetime"]
    + pd.to_timedelta(
        df["departure_hour"],
        unit="h"
    )
    + pd.to_timedelta(
        df["departure_minute"],
        unit="m"
    )
)

----------------------------TRIER----------------------------

In [116]:
df = df.sort_values(
    by=[
        "origin",
        "scheduled_departure_datetime"
    ]
)

-----------------------------------------------------CREER CONGESTION----------------------------

In [117]:
df = df.set_index(
    "scheduled_departure_datetime"
)
df["airport_traffic_30min"] = (

    df.groupby("origin")

    ["flight_number"]

    .rolling("30min")

    .count()

    .reset_index(level=0, drop=True)
)

---------------------------------VERIFIER--------------------------------------

In [118]:
print(
    df.reset_index()[
        [
            "scheduled_departure_datetime",
            "origin",
            "airport_traffic_30min"
        ]
    ].head(20)
)

   scheduled_departure_datetime origin  airport_traffic_30min
0           2008-01-04 04:00:00    ABE                    1.0
1           2008-01-04 04:00:00    ABE                    2.0
2           2008-01-04 04:00:00    ABE                    3.0
3           2008-01-04 04:15:00    ABE                    4.0
4           2008-01-04 04:40:00    ABE                    2.0
5           2008-01-04 05:20:00    ABE                    1.0
6           2008-01-04 06:28:00    ABE                    1.0
7           2008-01-04 07:25:00    ABE                    1.0
8           2008-01-04 07:50:00    ABE                    2.0
9           2008-01-04 08:50:00    ABE                    1.0
10          2008-01-04 10:15:00    ABE                    1.0
11          2008-01-04 10:40:00    ABE                    2.0
12          2008-01-04 11:00:00    ABE                    2.0
13          2008-01-04 11:01:00    ABE                    3.0
14          2008-01-11 04:00:00    ABE                    1.0
15      

---------------SAUVGARDER LES DONNEES-----------------------

In [119]:
df.to_parquet("featured_data.parquet", index=False)